# BETO + BiLSTM RF/RNF classifier
Train Spanish BETO with a bidirectional LSTM head. Labels: `0 = RF`, `1 = RNF`.

For Drive containing only `beto_rf_rnf.zip`, use `colab_beto_lstm_standalone.ipynb`. This repository-based notebook requires the full project folder on Drive.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
from pathlib import Path
from google.colab import files
import sys
import json
import random
import shutil
import subprocess
import zipfile

import numpy as np
import pandas as pd
import torch

DRIVE_ROOT = Path('/content/drive/MyDrive')
ROOT = DRIVE_ROOT / 'Protoype-Elbeto'
if not ROOT.exists():
    print('Project not found in Drive. Upload a ZIP of local Protoype-Elbeto folder.')
    uploaded = files.upload()
    zip_names = [name for name in uploaded if name.lower().endswith('.zip')]
    if not zip_names:
        raise FileNotFoundError('Upload project ZIP, for example Protoype-Elbeto.zip')
    staging = DRIVE_ROOT / '_elbeto_upload'
    staging.mkdir(exist_ok=True)
    with zipfile.ZipFile(zip_names[0]) as archive:
        archive.extractall(staging)
    etl_path = next(staging.rglob('etl_prepare.py'))
    source_root = etl_path.parent.parent
    shutil.copytree(source_root, ROOT, dirs_exist_ok=True)
    shutil.rmtree(staging, ignore_errors=True)
print('Project:', ROOT)
subprocess.run(['pip', '-q', 'install', '-r', str(ROOT / 'training/requirements-etl.txt'), 'accelerate'], check=True)
sys.path.insert(0, str(ROOT))
DATA_DIR = ROOT / 'training'
INIT_MODEL = ROOT / 'backend/models/beto_rf_rnf'
OUTPUT_DIR = ROOT / 'backend/models/beto_lstm_rf_rnf'
CHECKPOINT_DIR = ROOT / 'training/beto_lstm_checkpoints'
DATA_PREFIX = 'combined'
MAX_LENGTH = 128
BATCH_SIZE = 16
EPOCHS = 6
LEARNING_RATE = 2e-5
PATIENCE = 2
SEED = 42
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', DEVICE)

# If the initial artifact is only a ZIP in Drive, extract it without replacing an existing directory.
if not INIT_MODEL.exists():
    zip_candidates = [ROOT / 'beto_rf_rnf.zip', ROOT / 'models/beto_rf_rnf.zip']
    zip_path = next((path for path in zip_candidates if path.exists()), None)
    if zip_path is None:
        raise FileNotFoundError('Place beto_rf_rnf.zip in the project Drive folder.')
    with zipfile.ZipFile(zip_path) as archive:
        archive.extractall(ROOT)
assert INIT_MODEL.exists(), f'Missing initial BETO artifact: {INIT_MODEL}'
assert (DATA_DIR / 'Dataset6000Req.xlsx').exists(), 'Missing training/Dataset6000Req.xlsx'

## Prepare translated partitions
The source XLSX is English. This cell intentionally omits `--skip-translate`. Translation cache is stored on Drive. The first run may take time and can be resumed using the cache.

In [ ]:
etl_command = [
    'python', str(DATA_DIR / 'etl_prepare.py'),
    '--input', str(DATA_DIR / 'Dataset6000Req.xlsx'),
    '--additional-input', str(DATA_DIR / 'dataset_raw.csv'),
    '--output-dir', str(DATA_DIR),
    '--dataset-name', DATA_PREFIX,
    '--cache', str(DATA_DIR / '.translate_cache.json'),
]
subprocess.run(etl_command, cwd=ROOT, check=True)
report = json.loads((DATA_DIR / 'etl_report.json').read_text())
print(json.dumps(report, indent=2))
assert report['translated'] is True
print('Rows kept in original language after translation failures:', report['translation_fallback_rows'])

In [ ]:
from torch.utils.data import DataLoader
from transformers import AutoTokenizer, BertModel
from training.train_beto_lstm import (
    RequirementDataset, evaluate, load_partition, set_seed,
)
from backend.app.beto_lstm import BETOBiLSTM

set_seed(SEED)
frames = {name: load_partition(DATA_DIR, DATA_PREFIX, name) for name in ('train', 'val', 'test')}
print({name: frame.shape for name, frame in frames.items()})
print(frames['train']['label'].value_counts())

tokenizer = AutoTokenizer.from_pretrained(INIT_MODEL, local_files_only=True)
datasets = {name: RequirementDataset(frame, tokenizer, MAX_LENGTH) for name, frame in frames.items()}
loaders = {
    'train': DataLoader(datasets['train'], batch_size=BATCH_SIZE, shuffle=True),
    'val': DataLoader(datasets['val'], batch_size=BATCH_SIZE),
    'test': DataLoader(datasets['test'], batch_size=BATCH_SIZE),
}

In [ ]:
encoder = BertModel.from_pretrained(INIT_MODEL, local_files_only=True)
model = BETOBiLSTM(encoder, hidden_size=256, dropout=0.3).to(DEVICE)

# Preserve version 1 fine-tuning policy: freeze embeddings + first 9 BETO layers.
for parameter in model.encoder.embeddings.parameters():
    parameter.requires_grad = False
for layer in model.encoder.encoder.layer[:9]:
    for parameter in layer.parameters():
        parameter.requires_grad = False

trainable = sum(parameter.numel() for parameter in model.parameters() if parameter.requires_grad)
total = sum(parameter.numel() for parameter in model.parameters())
print(f'Trainable parameters: {trainable:,}/{total:,}')
optimizer = torch.optim.AdamW((p for p in model.parameters() if p.requires_grad), lr=LEARNING_RATE, weight_decay=0.01)
best_state = None
best_f1 = -1.0
stale_epochs = 0
history = []
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
for epoch in range(1, EPOCHS + 1):
    model.train()
    train_losses = []
    for batch in loaders['train']:
        batch = {key: value.to(DEVICE) for key, value in batch.items()}
        optimizer.zero_grad(set_to_none=True)
        output = model(**batch)
        output.loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        train_losses.append(float(output.loss.item()))
    validation = evaluate(model, loaders['val'], DEVICE)
    record = {'epoch': epoch, 'train_loss': float(np.mean(train_losses)), 'validation': validation}
    history.append(record)
    print(json.dumps(record, indent=2))
    torch.save({'model': model.state_dict(), 'optimizer': optimizer.state_dict(), 'epoch': epoch}, CHECKPOINT_DIR / f'epoch_{epoch}.pt')
    if validation['f1_macro'] > best_f1:
        best_f1 = validation['f1_macro']
        best_state = {key: value.detach().cpu().clone() for key, value in model.state_dict().items()}
        stale_epochs = 0
    else:
        stale_epochs += 1
        if stale_epochs >= PATIENCE:
            print('Early stopping')
            break

assert best_state is not None
model.load_state_dict(best_state)
test_metrics = evaluate(model, loaders['test'], DEVICE)
print('TEST METRICS')
print(json.dumps(test_metrics, indent=2))

In [ ]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
base_encoder_dir = OUTPUT_DIR / 'base_encoder'
model.encoder.save_pretrained(base_encoder_dir)
tokenizer.save_pretrained(OUTPUT_DIR)
head_state = {key: value.detach().cpu() for key, value in model.state_dict().items() if key.startswith(('lstm.', 'classifier.'))}
torch.save(head_state, OUTPUT_DIR / 'head.pt')
metadata = {
    'architecture': 'BETO+BiLSTM',
    'base_encoder': 'BETO',
    'labels': {'0': 'RF', '1': 'RNF'},
    'max_length': MAX_LENGTH,
    'lstm_hidden_size': 256,
    'dropout': 0.3,
    'data_prefix': DATA_PREFIX,
    'seed': SEED,
    'history': history,
    'test': test_metrics,
}
(OUTPUT_DIR / 'metadata.json').write_text(json.dumps(metadata, indent=2))
archive = shutil.make_archive(str(ROOT / 'beto_lstm_rf_rnf'), 'zip', root_dir=ROOT / 'backend/models', base_dir='beto_lstm_rf_rnf')
print('Saved artifact:', OUTPUT_DIR)
print('Saved archive:', archive)